# 11 — Export model và kiểm thử FastAPI
Phần mở rộng triển khai sau chuỗi báo cáo 01–08 của giảng viên.

In [ ]:
from pathlib import Path
import json, joblib
from fastapi.testclient import TestClient
import securaai_ml.api as api_module
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
api_module.ARTIFACT_DIR=ROOT/'artifacts'
api_module.load_artifacts.cache_clear()
app=api_module.app
model=joblib.load(ROOT/'artifacts'/'model.joblib')
metadata=json.loads((ROOT/'artifacts'/'model_metadata.json').read_text(encoding='utf-8'))
metadata

## Health check

In [ ]:
client=TestClient(app)
client.get('/health').json()

## Gửi một event chuẩn hóa

In [ ]:
payload={'timestamp':'2024-12-22T10:00:00Z','eventId':'1','providerName':'Microsoft-Windows-Sysmon','channel':'Microsoft-Windows-Sysmon/Operational','severityValue':'INFORMATION'}
response=client.post('/v1/predict',json=payload)
print(response.status_code)
response.json()

## Nguyên tắc tích hợp
Chạy shadow mode; lưu model version và evidence; không tự động đóng/mở incident chỉ từ score. Score không phải xác suất đã hiệu chỉnh.

## Cập nhật pipeline dữ liệu hợp nhất

Notebook này thuộc bộ thí nghiệm SecuraAI mới. Có hai tầng kết quả cần phân biệt:

- **Thí nghiệm giảng dạy 00–10:** so sánh thuật toán trên cùng split Windows-APT để bảo đảm công bằng.
- **Baseline hợp nhất theo từng event:** Windows-APT 2025 + RADAR + log Wazuh đã bộc lộ domain shift khi test trực tiếp.
- **Ứng viên temporal:** dùng `event_family`, `event_type` và thống kê cửa sổ 5 phút; vẫn chỉ chạy shadow.

RADAR bổ sung ground truth ransomware/Sysmon theo từng mẫu thực thi. Log Wazuh hiện tại chỉ là kiểm thử lab và weak-benign; không được trình bày như bằng chứng production. Test, threshold và dữ liệu ngoài miền được giữ tách biệt để giảm leakage.

In [ ]:
import pandas as pd

combined_artifact_dir = ROOT / 'artifacts-combined-candidate'
combined_metadata_path = combined_artifact_dir / 'model_metadata.json'
combined_source_report_path = ROOT / 'reports' / 'combined_candidate_by_source.json'
temporal_artifact_dir = ROOT / 'artifacts-temporal-candidate'
temporal_metadata_path = temporal_artifact_dir / 'model_metadata.json'

if combined_metadata_path.exists():
    combined_metadata = json.loads(combined_metadata_path.read_text(encoding='utf-8'))
    display(pd.DataFrame({
        'value': [
            combined_metadata['selectedModel'],
            combined_metadata['lockedThreshold'],
            ', '.join(combined_metadata['datasets']),
            combined_metadata['test']['f2'],
            combined_metadata['test']['precision'],
            combined_metadata['test']['recall'],
        ]
    }, index=['selected_model','locked_threshold','datasets','combined_test_f2','combined_test_precision','combined_test_recall']))
else:
    print('Chưa có artifacts-combined-candidate. Chạy scripts/train_combined_candidate.py trước.')

if temporal_metadata_path.exists():
    temporal_metadata = json.loads(temporal_metadata_path.read_text(encoding='utf-8'))
    display(pd.DataFrame({
        'value': [
            temporal_metadata['selectedModel'], temporal_metadata['lockedThreshold'],
            temporal_metadata['validation']['f2'], temporal_metadata['test']['f2'],
            temporal_metadata['test']['falsePositiveRate'], temporal_metadata['status'],
        ]
    }, index=['temporal_model','temporal_threshold','validation_f2','test_f2','test_fpr','status']))
else:
    print('Chưa có artifacts-temporal-candidate. Chạy scripts/train_temporal_candidate.py trước.')


## Export ứng viên temporal cho SecuraAI

Không ghi đè model production. API trỏ vào `artifacts-temporal-candidate` và backend gửi cả context năm phút. Model vẫn phải chạy shadow với contribution bằng 0.

In [ ]:
import securaai_ml.api as temporal_api
temporal_api.ARTIFACT_DIR = ROOT / 'artifacts-temporal-candidate'
temporal_api.load_artifacts.cache_clear()
temporal_client = TestClient(temporal_api.app)
health = temporal_client.get('/health')
print('HTTP', health.status_code)
display(health.json())


### Kiểm thử payload temporal

Payload phải có `eventFamily`, `eventType` và các bộ đếm cửa sổ năm phút. Score chỉ là điểm classifier, không phải xác suất tấn công.

In [ ]:
response=temporal_client.post('/v1/predict',json={
    'timestamp':'2026-11-03T02:00:00Z','eventFamily':'AUTHENTICATION',
    'eventType':'LOGIN_FAILURE','sourceIp':'203.0.113.10','status':'failure',
    'events5m':5,'failedLogins5m':5,'secondsSincePrevious':8,
})
print('HTTP',response.status_code); display(response.json())
assert response.status_code==200
